In [2]:
import os
import numpy as np
import faiss
import plotly.graph_objects as go
from scipy.io import loadmat

In [15]:
data_folder = "IITD_UPD"
folders = [f for f in os.listdir(data_folder) if os.path.isdir(os.path.join(data_folder, f))]

print(f"Number of folders inside IITD_UPD: {len(folders)}")

Number of folders inside IITD_UPD: 435


In [12]:


# ---------------------
# Load CASIA V1 Templates
# ---------------------
def load_templates(data_folder):
    """
    Load enrollment templates (excluding '_3') and probe templates (with '_3').
    Returns:
        index_vectors, index_labels, query_vectors, query_labels
    """
    files = sorted([f for f in os.listdir(data_folder) if f.endswith('.mat')])
    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    for file in files:
        path = os.path.join(data_folder, file)
        data = loadmat(path)
        vec = data['template'].flatten().astype('float32')

        # Each filename: e.g. 001_L_1.mat or 001_R_3.mat
        pid, side, _ = file.replace('.mat', '').split('_')
        label = f"{pid}_{side}"

        if '_3' in file:  # Probe (query)
            query_vectors.append(vec)
            query_labels.append(label)
        else:  # Enrollment (index)
            index_vectors.append(vec)
            index_labels.append(label)

    return (
        np.vstack(index_vectors),
        np.array(index_labels),
        np.vstack(query_vectors),
        np.array(query_labels)
    )

# ---------------------
# Build HNSW Index
# ---------------------
def build_hnsw_index(vectors, dim, M=16, efConstruction=200):
    index = faiss.IndexHNSWFlat(dim, M)
    index.hnsw.efConstruction = efConstruction
    index.add(vectors)
    return index

# ---------------------
# Evaluate Hit Rate
# ---------------------
def evaluate_hit_rate(index, index_labels, query_vectors, query_labels, ef_max=100):
    hit_rates = []
    x_labels = []
    total_indexed = len(index_labels)
    highlight_point = None

    for ef in range(1, ef_max + 1):
        index.hnsw.efSearch = ef
        distances, indices = index.search(query_vectors, k=1)
        predicted_labels = index_labels[indices.flatten()]
        hits = np.sum(predicted_labels == query_labels)
        total = len(query_labels)
        hit_rate = hits / total if total > 0 else 0
        hit_rates.append(hit_rate)
        penetration = ef / total_indexed
        x_labels.append(f"{ef} ({penetration:.2f})")

        if highlight_point is None and hit_rate >= 0.98:
            highlight_point = (ef, hit_rate, penetration)

        print(f"efSearch={ef}, Hit rate={hit_rate:.4f}")

    return x_labels, hit_rates, highlight_point

# ---------------------
# Plot Hit Rate
# ---------------------
def plot_hit_rate_plotly(x_labels, hit_rates, highlight_point):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=x_labels,
        y=hit_rates,
        mode='lines+markers',
        name="Hit Rate",
        line=dict(width=2),
        marker=dict(size=6)
    ))

    if highlight_point:
        ef_highlight, hr_highlight, penetration = highlight_point
        highlight_label = f"{ef_highlight} ({penetration:.2f})"
        fig.add_trace(go.Scatter(
            x=[highlight_label],
            y=[hr_highlight],
            mode='markers+text',
            marker=dict(color='red', size=12, symbol='star'),
            text=[f"Penetration={penetration:.2f}"],
            textposition="top center",
            name="First ≥98% Hit"
        ))

    fig.update_layout(
        title="CASIAV1_HNSW",
        xaxis_title="efSearch (Penetration)",
        yaxis_title="Hit Rate",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()

# ---------------------
# Main Workflow
# ---------------------
def main(data_folder, ef_max=50):
    index_x, index_y, query_x, query_y = load_templates(data_folder)
    dim = index_x.shape[1]
    print(f"Loaded {len(index_y)} index templates and {len(query_y)} queries with dim={dim}")

    index = build_hnsw_index(index_x, dim)
    x_labels, hit_rates, highlight_point = evaluate_hit_rate(
        index, index_y, query_x, query_y, ef_max=ef_max
    )

    plot_hit_rate_plotly(x_labels, hit_rates, highlight_point)

# Example usage
if __name__ == "__main__":
    main(data_folder, ef_max=50)


Loaded 216 index templates and 108 queries with dim=9600
efSearch=1, Hit rate=0.6481
efSearch=2, Hit rate=0.7778
efSearch=3, Hit rate=0.8241
efSearch=4, Hit rate=0.8426
efSearch=5, Hit rate=0.8704
efSearch=6, Hit rate=0.8796
efSearch=7, Hit rate=0.8889
efSearch=8, Hit rate=0.8796
efSearch=9, Hit rate=0.8889
efSearch=10, Hit rate=0.8981
efSearch=11, Hit rate=0.8981
efSearch=12, Hit rate=0.8981
efSearch=13, Hit rate=0.8981
efSearch=14, Hit rate=0.9074
efSearch=15, Hit rate=0.9074
efSearch=16, Hit rate=0.9074
efSearch=17, Hit rate=0.9074
efSearch=18, Hit rate=0.9074
efSearch=19, Hit rate=0.9074
efSearch=20, Hit rate=0.9074
efSearch=21, Hit rate=0.9074
efSearch=22, Hit rate=0.9074
efSearch=23, Hit rate=0.9074
efSearch=24, Hit rate=0.9074
efSearch=25, Hit rate=0.9074
efSearch=26, Hit rate=0.9074
efSearch=27, Hit rate=0.9074
efSearch=28, Hit rate=0.9074
efSearch=29, Hit rate=0.9074
efSearch=30, Hit rate=0.9074
efSearch=31, Hit rate=0.9074
efSearch=32, Hit rate=0.9074
efSearch=33, Hit rate=0.

In [4]:
import os
import numpy as np
import random
import faiss
import plotly.graph_objects as go

# ---------------------
# Load IITD_UPD Templates (70:30 Split)
# ---------------------
def load_iitd_upd_templates(root_folder, seed=42, split_ratio=0.7):
    """
    For each subject folder (like 1_L, 2_L...), randomly split its .npy files
      - 70% for indexing (gallery/enrollment)
      - 30% for query (probe)
    Returns:
        index_vectors, index_labels, query_vectors, query_labels
    """
    random.seed(seed)
    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    subject_folders = sorted(
        [d for d in os.listdir(root_folder) if os.path.isdir(os.path.join(root_folder, d))]
    )

    for subject in subject_folders:
        subject_path = os.path.join(root_folder, subject)
        files = sorted([f for f in os.listdir(subject_path) if f.endswith('.npy')])

        n = len(files)
        if n == 0:
            print(f"⚠️ Skipping {subject}, no .npy files found.")
            continue

        random.shuffle(files)
        split_point = max(1, int(n * split_ratio))  # At least 1 for indexing
        index_files = files[:split_point]
        query_files = files[split_point:] if split_point < n else []

        # Load index vectors
        for f in index_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            index_vectors.append(vec)
            index_labels.append(subject)

        # Load query vectors
        for f in query_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            query_vectors.append(vec)
            query_labels.append(subject)

        print(f"{subject}: {len(index_files)} index, {len(query_files)} query")

    return (
        np.vstack(index_vectors),
        np.array(index_labels),
        np.vstack(query_vectors),
        np.array(query_labels)
    )

# ---------------------
# Build HNSW Index
# ---------------------
def build_hnsw_index(vectors, dim, M=16, efConstruction=200):
    index = faiss.IndexHNSWFlat(dim, M)
    index.hnsw.efConstruction = efConstruction
    index.add(vectors)
    return index

# ---------------------
# Evaluate Hit Rate
# ---------------------
def evaluate_hit_rate(index, index_labels, query_vectors, query_labels, ef_max=100):
    hit_rates = []
    x_labels = []
    total_indexed = len(index_labels)
    highlight_point = None

    for ef in range(1, ef_max + 1):
        index.hnsw.efSearch = ef
        distances, indices = index.search(query_vectors, k=1)
        predicted_labels = index_labels[indices.flatten()]

        hits = np.sum(predicted_labels == query_labels)
        total = len(query_labels)
        hit_rate = hits / total if total > 0 else 0
        hit_rates.append(hit_rate)
        penetration = ef / total_indexed
        x_labels.append(f"{ef} ({penetration:.2f})")

        if highlight_point is None and hit_rate >= 0.98:
            highlight_point = (ef, hit_rate, penetration)

        print(f"efSearch={ef}, Hit rate={hit_rate:.4f}")

    return x_labels, hit_rates, highlight_point

# ---------------------
# Plot Hit Rate
# ---------------------
def plot_hit_rate_plotly(x_labels, hit_rates, highlight_point):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_labels,
        y=hit_rates,
        mode='lines+markers',
        name="Hit Rate",
        line=dict(width=2),
        marker=dict(size=6)
    ))

    if highlight_point:
        ef_highlight, hr_highlight, penetration = highlight_point
        highlight_label = f"{ef_highlight} ({penetration:.2f})"
        fig.add_trace(go.Scatter(
            x=[highlight_label],
            y=[hr_highlight],
            mode='markers+text',
            marker=dict(color='red', size=12, symbol='star'),
            text=[f"Penetration={penetration:.2f}"],
            textposition="top center",
            name="First ≥98% Hit"
        ))

    fig.update_layout(
        title="IITD_UPD_HNSW (70:30 Split)",
        xaxis_title="efSearch (Penetration)",
        yaxis_title="Hit Rate",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()

# ---------------------
# Main Workflow
# ---------------------
def main(data_folder, ef_max=50):
    index_x, index_y, query_x, query_y = load_iitd_upd_templates(data_folder)
    dim = index_x.shape[1]
    print(f"\nLoaded {len(index_y)} index templates and {len(query_y)} queries with dim={dim}")

    index = build_hnsw_index(index_x, dim)
    x_labels, hit_rates, highlight_point = evaluate_hit_rate(
        index, index_y, query_x, query_y, ef_max=ef_max
    )

    plot_hit_rate_plotly(x_labels, hit_rates, highlight_point)

# ---------------------
# Example Usage
# ---------------------
if __name__ == "__main__":
    data_folder = "IITD_UPD"
    main(data_folder, ef_max=50)


100_L: 3 index, 2 query
100_R: 3 index, 2 query
101_L: 3 index, 2 query
101_R: 3 index, 2 query
102_L: 3 index, 2 query
102_R: 3 index, 2 query
103_L: 3 index, 2 query
103_R: 3 index, 2 query
104_L: 3 index, 2 query
104_R: 3 index, 2 query
105_L: 3 index, 2 query
105_R: 3 index, 2 query
106_L: 3 index, 2 query
106_R: 3 index, 2 query
107_L: 3 index, 2 query
107_R: 3 index, 2 query
108_L: 3 index, 2 query
108_R: 3 index, 2 query
109_L: 3 index, 2 query
109_R: 3 index, 2 query
10_L: 7 index, 3 query
110_L: 3 index, 2 query
110_R: 3 index, 2 query
111_L: 3 index, 2 query
111_R: 3 index, 2 query
112_L: 3 index, 2 query
112_R: 3 index, 2 query
113_L: 3 index, 2 query
113_R: 3 index, 2 query
114_L: 3 index, 2 query
114_R: 3 index, 2 query
115_L: 3 index, 2 query
115_R: 3 index, 2 query
116_L: 3 index, 2 query
116_R: 3 index, 2 query
117_L: 3 index, 2 query
117_R: 3 index, 2 query
118_L: 3 index, 2 query
118_R: 3 index, 2 query
119_L: 3 index, 2 query
119_R: 3 index, 2 query
11_L: 7 index, 3 

In [8]:
import os
import numpy as np
import random
import faiss
import plotly.graph_objects as go

# =========================================================
#  Load IITD_UPD Templates (70:30 Split)
# =========================================================
def load_iitd_upd_templates(root_folder, seed=42, split_ratio=0.7):
    """
    For each subject folder (like 1_L, 2_L...), randomly split its .npy files
      - 70% for indexing (gallery/enrollment)
      - 30% for query (probe)
    Returns:
        index_vectors, index_labels, query_vectors, query_labels
    """
    random.seed(seed)
    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    subject_folders = sorted(
        [d for d in os.listdir(root_folder) if os.path.isdir(os.path.join(root_folder, d))]
    )

    for subject in subject_folders:
        subject_path = os.path.join(root_folder, subject)
        files = sorted([f for f in os.listdir(subject_path) if f.endswith('.npy')])

        n = len(files)
        if n == 0:
            print(f"⚠️ Skipping {subject}, no .npy files found.")
            continue

        random.shuffle(files)
        split_point = max(1, int(n * split_ratio))  # At least one for index
        index_files = files[:split_point]
        query_files = files[split_point:] if split_point < n else []

        for f in index_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            index_vectors.append(vec)
            index_labels.append(subject)

        for f in query_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            query_vectors.append(vec)
            query_labels.append(subject)

        print(f"{subject}: {len(index_files)} index, {len(query_files)} query")

    return (
        np.vstack(index_vectors),
        np.array(index_labels),
        np.vstack(query_vectors),
        np.array(query_labels)
    )


# =========================================================
#  Build HNSW Index
# =========================================================
def build_hnsw_index(vectors, dim, M=16, efConstruction=200, efSearch=20):
    index = faiss.IndexHNSWFlat(dim, M)
    index.hnsw.efConstruction = efConstruction
    index.hnsw.efSearch = efSearch
    index.add(vectors)
    return index


# =========================================================
#  Evaluate Hit Rate vs Penetration Rate
# =========================================================
def evaluate_hit_rate_topk(index, index_labels, query_vectors, query_labels, k_max=50):
    total_indexed = len(index_labels)
    total_queries = len(query_labels)
    hit_rates = []
    penetration_rates = []

    # Retrieve all neighbors up to k_max
    distances, indices = index.search(query_vectors, k=k_max)
    predicted_labels = index_labels[indices]

    for k in range(1, k_max + 1):
        hits = sum(
            query_labels[i] in predicted_labels[i, :k]
            for i in range(total_queries)
        )
        hit_rate = hits / total_queries
        penetration = k / total_indexed
        hit_rates.append(hit_rate)
        penetration_rates.append(penetration)
        print(f"Top-{k}: Hit rate={hit_rate:.4f}, Penetration={penetration:.4f}")

    return penetration_rates, hit_rates


# =========================================================
#  Plot Hit Rate vs Penetration Rate
# =========================================================
def plot_hit_rate_vs_penetration(penetration_rates, hit_rates, efSearch):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=penetration_rates,
        y=hit_rates,
        mode='lines+markers',
        name=f"efSearch={efSearch}",
        line=dict(width=2),
        marker=dict(size=6)
    ))

    # Clean tick marks at 0.01 intervals
    tick_vals = np.arange(0.0, max(penetration_rates) + 0.01, 0.01)
    fig.update_xaxes(
        title='Penetration Rate (k / total indexed)',
        tickvals=tick_vals,
        tickformat=".2f"
    )
    fig.update_yaxes(title='Hit Rate')

    fig.update_layout(
        title=f"IITD_UPD_HNSW: Hit Rate vs Penetration Rate (efSearch={efSearch})",
        template='plotly_white',
        width=800,
        height=500,
        hovermode="x unified"
    )
    fig.show()


# =========================================================
#  Main Workflow
# =========================================================
def main(data_folder, efSearch=20, k_max=50):
    index_x, index_y, query_x, query_y = load_iitd_upd_templates(data_folder)
    dim = index_x.shape[1]
    print(f"\nLoaded {len(index_y)} index templates and {len(query_y)} queries with dim={dim}")

    index = build_hnsw_index(index_x, dim, efSearch=efSearch)
    penetration_rates, hit_rates = evaluate_hit_rate_topk(index, index_y, query_x, query_y, k_max=k_max)
    plot_hit_rate_vs_penetration(penetration_rates, hit_rates, efSearch)


# =========================================================
#  Example Usage
# =========================================================
if __name__ == "__main__":
    data_folder = "IITD_UPD"   # path to your IITD_UPD dataset
    main(data_folder, efSearch=20, k_max=50)


100_L: 3 index, 2 query
100_R: 3 index, 2 query
101_L: 3 index, 2 query
101_R: 3 index, 2 query
102_L: 3 index, 2 query
102_R: 3 index, 2 query
103_L: 3 index, 2 query
103_R: 3 index, 2 query
104_L: 3 index, 2 query
104_R: 3 index, 2 query
105_L: 3 index, 2 query
105_R: 3 index, 2 query
106_L: 3 index, 2 query
106_R: 3 index, 2 query
107_L: 3 index, 2 query
107_R: 3 index, 2 query
108_L: 3 index, 2 query
108_R: 3 index, 2 query
109_L: 3 index, 2 query
109_R: 3 index, 2 query
10_L: 7 index, 3 query
110_L: 3 index, 2 query
110_R: 3 index, 2 query
111_L: 3 index, 2 query
111_R: 3 index, 2 query
112_L: 3 index, 2 query
112_R: 3 index, 2 query
113_L: 3 index, 2 query
113_R: 3 index, 2 query
114_L: 3 index, 2 query
114_R: 3 index, 2 query
115_L: 3 index, 2 query
115_R: 3 index, 2 query
116_L: 3 index, 2 query
116_R: 3 index, 2 query
117_L: 3 index, 2 query
117_R: 3 index, 2 query
118_L: 3 index, 2 query
118_R: 3 index, 2 query
119_L: 3 index, 2 query
119_R: 3 index, 2 query
11_L: 7 index, 3 

In [9]:
import os
import numpy as np
import random
import faiss
import plotly.graph_objects as go

# =========================================================
#  Load IITD_UPD Templates (70:30 Split)
# =========================================================
def load_iitd_upd_templates(root_folder, seed=42, split_ratio=0.7):
    random.seed(seed)
    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    subject_folders = sorted(
        [d for d in os.listdir(root_folder) if os.path.isdir(os.path.join(root_folder, d))]
    )

    for subject in subject_folders:
        subject_path = os.path.join(root_folder, subject)
        files = sorted([f for f in os.listdir(subject_path) if f.endswith('.npy')])
        if not files:
            continue

        random.shuffle(files)
        split_point = max(1, int(len(files) * split_ratio))
        index_files = files[:split_point]
        query_files = files[split_point:]

        for f in index_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            index_vectors.append(vec)
            index_labels.append(subject)

        for f in query_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            query_vectors.append(vec)
            query_labels.append(subject)

    return (
        np.vstack(index_vectors),
        np.array(index_labels),
        np.vstack(query_vectors),
        np.array(query_labels)
    )


# =========================================================
#  Hybrid Clustering + HNSW Index
# =========================================================
class HybridClusterHNSW:
    def __init__(self, dim, n_clusters=10, M=16, efC=200, efS=20):
        self.dim = dim
        self.n_clusters = n_clusters
        self.M = M
        self.efC = efC
        self.efS = efS
        self.kmeans = faiss.Kmeans(dim, n_clusters, niter=20, verbose=False)
        self.cluster_indices = []   # list of faiss.IndexHNSWFlat
        self.cluster_labels = []    # parallel list of labels for each index

    def fit(self, vectors, labels):
        print("Performing coarse clustering...")
        self.kmeans.train(vectors)
        _, assign = self.kmeans.index.search(vectors, 1)
        assign = assign.flatten()

        self.cluster_indices = []
        self.cluster_labels = []

        for c in range(self.n_clusters):
            members = np.where(assign == c)[0]
            if len(members) == 0:
                self.cluster_indices.append(None)
                self.cluster_labels.append(None)
                continue

            cluster_vecs = vectors[members]
            cluster_labs = labels[members]

            index = faiss.IndexHNSWFlat(self.dim, self.M)
            index.hnsw.efConstruction = self.efC
            index.hnsw.efSearch = self.efS
            index.add(cluster_vecs)

            self.cluster_indices.append(index)
            self.cluster_labels.append(cluster_labs)

            print(f"Cluster {c}: {len(members)} vectors")

    def search(self, queries, top_clusters=3, k=10):
        """Search queries: first cluster search, then within top clusters' HNSW"""
        # Step 1: find nearest clusters
        _, cluster_ids = self.kmeans.index.search(queries, top_clusters)
        all_labels = []
        for qi, q in enumerate(queries):
            labels_collected = []
            for cid in cluster_ids[qi]:
                if self.cluster_indices[cid] is None:
                    continue
                index = self.cluster_indices[cid]
                labs = self.cluster_labels[cid]
                D, I = index.search(q.reshape(1, -1), k)
                labels_collected.extend(labs[I.flatten()])
            all_labels.append(labels_collected)
        return all_labels


# =========================================================
#  Evaluation (Hit Rate vs Penetration)
# =========================================================
def evaluate_hit_rate_topk(hybrid_index, index_labels, query_vectors, query_labels, k_max=50):
    total_indexed = len(index_labels)
    total_queries = len(query_labels)
    hit_rates, penetration_rates = [], []

    for k in range(1, k_max + 1):
        hits = 0
        retrieved_total = 0
        predicted_labels_all = hybrid_index.search(query_vectors, top_clusters=3, k=k)
        for qi, predicted_labels in enumerate(predicted_labels_all):
            retrieved_total += len(predicted_labels)
            if query_labels[qi] in predicted_labels[:k]:
                hits += 1
        hit_rate = hits / total_queries
        penetration = min(k * hybrid_index.n_clusters, total_indexed) / total_indexed
        hit_rates.append(hit_rate)
        penetration_rates.append(penetration)
        print(f"Top-{k}: Hit rate={hit_rate:.4f}, Penetration={penetration:.4f}")
    return penetration_rates, hit_rates


# =========================================================
#  Plotting
# =========================================================
def plot_hit_rate_vs_penetration(penetration_rates, hit_rates, efSearch):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=penetration_rates,
        y=hit_rates,
        mode='lines+markers',
        name=f"Hybrid efSearch={efSearch}",
        line=dict(width=2),
        marker=dict(size=6)
    ))
    fig.update_layout(
        title=f"Hybrid Clustered-HNSW: Hit Rate vs Penetration (efSearch={efSearch})",
        xaxis_title="Penetration Rate",
        yaxis_title="Hit Rate",
        template="plotly_white"
    )
    fig.show()


# =========================================================
#  Main
# =========================================================
def main(data_folder, n_clusters=10, efSearch=20, k_max=50):
    index_x, index_y, query_x, query_y = load_iitd_upd_templates(data_folder)
    dim = index_x.shape[1]
    print(f"\nLoaded {len(index_y)} index templates and {len(query_y)} queries, dim={dim}")

    hybrid = HybridClusterHNSW(dim, n_clusters=n_clusters, efS=efSearch)
    hybrid.fit(index_x, index_y)
    penetration_rates, hit_rates = evaluate_hit_rate_topk(hybrid, index_y, query_x, query_y, k_max=k_max)
    plot_hit_rate_vs_penetration(penetration_rates, hit_rates, efSearch)


if __name__ == "__main__":
    data_folder = "IITD_UPD"
    main(data_folder, n_clusters=10, efSearch=20, k_max=50)



Loaded 1357 index templates and 883 queries, dim=1024
Performing coarse clustering...
Cluster 0: 165 vectors
Cluster 1: 167 vectors
Cluster 2: 125 vectors
Cluster 3: 156 vectors
Cluster 4: 104 vectors
Cluster 5: 122 vectors
Cluster 6: 104 vectors
Cluster 7: 177 vectors
Cluster 8: 152 vectors
Cluster 9: 85 vectors
Top-1: Hit rate=0.7792, Penetration=0.0074
Top-2: Hit rate=0.8029, Penetration=0.0147
Top-3: Hit rate=0.8120, Penetration=0.0221
Top-4: Hit rate=0.8211, Penetration=0.0295
Top-5: Hit rate=0.8256, Penetration=0.0368
Top-6: Hit rate=0.8290, Penetration=0.0442
Top-7: Hit rate=0.8324, Penetration=0.0516
Top-8: Hit rate=0.8347, Penetration=0.0590
Top-9: Hit rate=0.8369, Penetration=0.0663
Top-10: Hit rate=0.8369, Penetration=0.0737
Top-11: Hit rate=0.8369, Penetration=0.0811
Top-12: Hit rate=0.8369, Penetration=0.0884
Top-13: Hit rate=0.8369, Penetration=0.0958
Top-14: Hit rate=0.8369, Penetration=0.1032
Top-15: Hit rate=0.8392, Penetration=0.1105
Top-16: Hit rate=0.8403, Penetrat

In [12]:
import os
import numpy as np
import random
import faiss
import plotly.graph_objects as go
from collections import defaultdict
import heapq

# ---------------------------
# Load IITD_UPD Templates (70:30 Split)
# ---------------------------
def load_iitd_upd_templates(root_folder, seed=42, split_ratio=0.7):
    random.seed(seed)
    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    subject_folders = sorted(
        [d for d in os.listdir(root_folder) if os.path.isdir(os.path.join(root_folder, d))]
    )

    for subject in subject_folders:
        subject_path = os.path.join(root_folder, subject)
        files = sorted([f for f in os.listdir(subject_path) if f.endswith('.npy')])
        if not files:
            print(f"⚠️ Skipping {subject}, no .npy files")
            continue

        random.shuffle(files)
        split_point = max(1, int(len(files) * split_ratio))
        index_files = files[:split_point]
        query_files = files[split_point:]

        for f in index_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            index_vectors.append(vec)
            index_labels.append(subject)

        for f in query_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            query_vectors.append(vec)
            query_labels.append(subject)

        print(f"{subject}: {len(index_files)} index, {len(query_files)} query")

    return (
        np.vstack(index_vectors),
        np.array(index_labels),
        np.vstack(query_vectors),
        np.array(query_labels)
    )

# ---------------------------
# LSH helper: random projection hash family
# ---------------------------
class LSHTable:
    def __init__(self, dim, bits, rng):
        """
        bits: number of random projection bits (hash length)
        rng: numpy RandomState
        """
        self.bits = bits
        self.dim = dim
        # random projection vectors shape (bits, dim)
        self.projections = rng.normal(size=(bits, dim)).astype('float32')
        # dictionary: bucket_key (tuple of bits) -> list of (local_index)
        self.buckets = defaultdict(list)

    def compute_hash(self, vecs):
        # vecs: N x dim -> returns N x bits of 0/1
        dots = np.dot(vecs, self.projections.T)  # N x bits
        bits_mat = (dots >= 0).astype('int8')
        # convert to tuple of ints per vector for use as dict key
        keys = [tuple(row.tolist()) for row in bits_mat]
        return keys

    def add(self, vecs, local_indices):
        keys = self.compute_hash(vecs)
        for key, idx in zip(keys, local_indices):
            self.buckets[key].append(idx)

    def query_bucket(self, vec):
        key = tuple((np.dot(vec, self.projections.T) >= 0).astype('int8').tolist())
        return self.buckets.get(key, [])

# ---------------------------
# Hybrid: KMeans clustering + LSH per cluster
# ---------------------------
class HybridClusterLSH:
    def __init__(self, dim, n_clusters=10, L=5, bits=16, rng_seed=42):
        """
        n_clusters: number of coarse clusters
        L: number of LSH tables per cluster
        bits: number of projection bits per table
        """
        self.dim = dim
        self.n_clusters = n_clusters
        self.L = L
        self.bits = bits
        self.rng = np.random.RandomState(rng_seed)
        self.kmeans = faiss.Kmeans(dim, n_clusters, niter=20, verbose=False)
        # For each cluster: store dict with 'indices' (global indices), 'labels', 'vecs', 'lsh_tables' (list)
        self.clusters = [None] * n_clusters

    def fit(self, vectors, labels):
        """
        vectors: N x dim (float32)
        labels: N (object) or array
        We will keep the mapping for global indices so we can compute penetration easily.
        """
        print("Training coarse KMeans...")
        self.kmeans.train(vectors)
        _, assign = self.kmeans.index.search(vectors, 1)
        assign = assign.flatten()

        # build clusters
        for c in range(self.n_clusters):
            members = np.where(assign == c)[0]
            if len(members) == 0:
                self.clusters[c] = None
                print(f"Cluster {c}: EMPTY")
                continue

            cluster_vecs = vectors[members]
            cluster_labels = labels[members]
            # Build L LSH tables for this cluster
            lsh_tables = []
            for i in range(self.L):
                lt = LSHTable(self.dim, self.bits, rng=self.rng)
                lt.add(cluster_vecs, list(range(len(members))))  # local indices 0..len(members)-1
                lsh_tables.append(lt)

            self.clusters[c] = {
                'global_indices': members,      # map local index -> global index by members[local]
                'vecs': cluster_vecs,          # local vecs
                'labels': cluster_labels,
                'lsh_tables': lsh_tables
            }
            print(f"Cluster {c}: {len(members)} vectors, built {self.L} LSH tables")

    def _probe_cluster(self, cluster, query_vec, probes_per_table=2):
        """
        Probe one cluster for one query:
        - For each LSH table, fetch the bucket for the query (single bucket).
        - Optionally we could also probe nearby buckets, but here keep single-bucket per table.
        - Return set of global indices (deduped)
        """
        candidates_local = set()
        for lt in cluster['lsh_tables']:
            bucket = lt.query_bucket(query_vec)
            # bucket contains local indices; add up to probes_per_table of them (or all)
            # choose first probes_per_table (deterministic); could random.sample for randomness
            for li in bucket[:probes_per_table]:
                candidates_local.add(li)
        # map local -> global
        global_idxs = [int(cluster['global_indices'][li]) for li in candidates_local]
        return global_idxs

    def search(self, queries, top_clusters=3, k=10, probes_per_table=2):
        """
        queries: Q x dim
        returns for each query a list of candidate global indices (unsorted)
        """
        # Step 1: coarse assign top_clusters based on centroid distances
        # use kmeans centroids: self.kmeans.centroids (n_clusters x dim)
        # FAISS stores centroids in self.kmeans.centroids
        centroids = np.array(self.kmeans.centroids).astype('float32')
        # compute centroid distances for all queries
        # queries shape Qxdim
        dists = np.linalg.norm(queries[:, None, :] - centroids[None, :, :], axis=2)  # Q x n_clusters
        top_cluster_ids = np.argsort(dists, axis=1)[:, :top_clusters]

        all_candidates = []
        for qi, q in enumerate(queries):
            qvec = q.astype('float32')
            candidate_globals = []
            for cid in top_cluster_ids[qi]:
                cluster = self.clusters[cid]
                if cluster is None:
                    continue
                cand = self._probe_cluster(cluster, qvec, probes_per_table=probes_per_table)
                candidate_globals.extend(cand)
            # dedupe
            candidate_globals = list(dict.fromkeys(candidate_globals))
            all_candidates.append(candidate_globals)
        return all_candidates

    def rerank_and_get_topk(self, queries, candidates_list, vectors, k=10):
        """
        For each query compute distances to candidate global indices (exact) and return top-k global indices.
        vectors: full database vectors (N x dim) for distance computation
        """
        results = []
        for qi, cand in enumerate(candidates_list):
            if len(cand) == 0:
                results.append([])
                continue
            cand_vecs = vectors[cand]  # len(cand) x dim
            qvec = queries[qi].reshape(1, -1)
            dists = np.linalg.norm(cand_vecs - qvec, axis=1)  # euclidean
            # get top k smallest distances
            topk_idx = np.argsort(dists)[:k]
            topk_global = [cand[i] for i in topk_idx]
            results.append(topk_global)
        return results

# ---------------------------
# Evaluate Hit Rate vs Penetration Rate
# ---------------------------
def evaluate_hit_rate_topk(hybrid_index, full_vectors, index_labels, query_vectors, query_labels,
                           k_max=50, top_clusters=3, probes_per_table=2):
    total_indexed = len(index_labels)
    total_queries = len(query_labels)

    hit_rates = []
    penetration_rates = []

    # Precompute candidate sets for various k? We'll compute per k by reranking top-k.
    # But candidates depend on probes per table and top_clusters only, not k. So compute once:
    candidates_all = hybrid_index.search(query_vectors, top_clusters=top_clusters, k=k_max, probes_per_table=probes_per_table)

    # For incremental k we will rerank and check top-k from reranked candidates
    reranked_topk_all = hybrid_index.rerank_and_get_topk(query_vectors, candidates_all, full_vectors, k=k_max)

    for k in range(1, k_max + 1):
        hits = 0
        for qi in range(total_queries):
            topk = reranked_topk_all[qi][:k]
            # topk contains global indices -> map to labels using index_labels
            if len(topk) > 0:
                topk_labels = index_labels[topk]
                if query_labels[qi] in topk_labels:
                    hits += 1
        hit_rate = hits / total_queries
        # penetration = average retrieved candidates per query divided by total indexed
        avg_retrieved = np.mean([len(reranked_topk_all[qi][:k]) for qi in range(total_queries)])
        # but more meaningful penetration is k / total_indexed (like before). We'll keep that consistent:
        penetration = k / total_indexed
        hit_rates.append(hit_rate)
        penetration_rates.append(penetration)
        print(f"Top-{k}: Hit rate={hit_rate:.4f}, Penetration={penetration:.4f}")
    return penetration_rates, hit_rates

# ---------------------------
# Plotter
# ---------------------------
def plot_hit_rate_vs_penetration(penetration_rates, hit_rates, title):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=penetration_rates,
        y=hit_rates,
        mode='lines+markers',
        name='Cluster+LSH',
        line=dict(width=2),
        marker=dict(size=6)
    ))
    tick_vals = np.arange(0.0, max(penetration_rates) + 0.01, 0.01)
    fig.update_xaxes(title='Penetration Rate (k / total indexed)', tickvals=tick_vals, tickformat=".2f")
    fig.update_yaxes(title='Hit Rate')
    fig.update_layout(title=title, template='plotly_white', width=800, height=500, hovermode='x unified')
    fig.show()

# ---------------------------
# Main workflow
# ---------------------------
def main(data_folder,
         n_clusters=10,
         L=5,
         bits=16,
         top_clusters=3,
         probes_per_table=2,
         k_max=50,
         rng_seed=42):
    index_x, index_y, query_x, query_y = load_iitd_upd_templates(data_folder, seed=rng_seed)
    dim = index_x.shape[1]
    print(f"\nLoaded {len(index_y)} index templates and {len(query_y)} queries with dim={dim}")

    hybrid = HybridClusterLSH(dim, n_clusters=n_clusters, L=L, bits=bits, rng_seed=rng_seed)
    hybrid.fit(index_x, index_y)

    penetration_rates, hit_rates = evaluate_hit_rate_topk(
        hybrid, index_x, index_y, query_x, query_y,
        k_max=k_max, top_clusters=top_clusters, probes_per_table=probes_per_table
    )

    plot_hit_rate_vs_penetration(penetration_rates, hit_rates,
                                 title=f"IITD_UPD: Clustered + LSH (n_clusters={n_clusters}, L={L}, bits={bits}, top_clusters={top_clusters})")

if __name__ == "__main__":
    data_folder = "IITD_UPD"
    main(data_folder,
         n_clusters=12,
         L=6,
         bits=16,
         top_clusters=6,
         probes_per_table=3,
         k_max=50,
         rng_seed=1234)


100_L: 3 index, 2 query
100_R: 3 index, 2 query
101_L: 3 index, 2 query
101_R: 3 index, 2 query
102_L: 3 index, 2 query
102_R: 3 index, 2 query
103_L: 3 index, 2 query
103_R: 3 index, 2 query
104_L: 3 index, 2 query
104_R: 3 index, 2 query
105_L: 3 index, 2 query
105_R: 3 index, 2 query
106_L: 3 index, 2 query
106_R: 3 index, 2 query
107_L: 3 index, 2 query
107_R: 3 index, 2 query
108_L: 3 index, 2 query
108_R: 3 index, 2 query
109_L: 3 index, 2 query
109_R: 3 index, 2 query
10_L: 7 index, 3 query
110_L: 3 index, 2 query
110_R: 3 index, 2 query
111_L: 3 index, 2 query
111_R: 3 index, 2 query
112_L: 3 index, 2 query
112_R: 3 index, 2 query
113_L: 3 index, 2 query
113_R: 3 index, 2 query
114_L: 3 index, 2 query
114_R: 3 index, 2 query
115_L: 3 index, 2 query
115_R: 3 index, 2 query
116_L: 3 index, 2 query
116_R: 3 index, 2 query
117_L: 3 index, 2 query
117_R: 3 index, 2 query
118_L: 3 index, 2 query
118_R: 3 index, 2 query
119_L: 3 index, 2 query
119_R: 3 index, 2 query
11_L: 7 index, 3 

In [16]:
import os
import numpy as np
import random
import faiss
from sklearn.cluster import KMeans, AgglomerativeClustering
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')


# ------------------------------------------------------------
# 1. Load IITD_UPD Templates with 70:30 Split per Subject
# ------------------------------------------------------------
def load_iitd_upd_templates(root_folder, seed=42, split_ratio=0.7):
    random.seed(seed)
    index_vectors, index_labels, query_vectors, query_labels = [], [], [], []

    subject_folders = sorted(
        [d for d in os.listdir(root_folder) if os.path.isdir(os.path.join(root_folder, d))]
    )

    for subject in subject_folders:
        subject_path = os.path.join(root_folder, subject)
        files = sorted([f for f in os.listdir(subject_path) if f.endswith('.npy')])

        if len(files) == 0:
            continue

        random.shuffle(files)
        split_point = max(1, int(len(files) * split_ratio))
        index_files = files[:split_point]
        query_files = files[split_point:]

        for f in index_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            index_vectors.append(vec)
            index_labels.append(subject)

        for f in query_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            query_vectors.append(vec)
            query_labels.append(subject)

    print(f"✅ Loaded {len(index_labels)} index and {len(query_labels)} query samples "
          f"from {len(subject_folders)} subjects.")
    return (
        np.vstack(index_vectors),
        np.array(index_labels),
        np.vstack(query_vectors),
        np.array(query_labels)
    )


# ------------------------------------------------------------
# 2. Build HNSW Index
# ------------------------------------------------------------
def build_hnsw_index(vectors, dim, M=16, efConstruction=200, efSearch=20):
    index = faiss.IndexHNSWFlat(dim, M)
    index.hnsw.efConstruction = efConstruction
    index.hnsw.efSearch = efSearch
    index.add(vectors)
    return index


# ------------------------------------------------------------
# 3. Build KMeans / Agglomerative Index Tables
# ------------------------------------------------------------
def build_kmeans_index(gallery_features, n_clusters):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_assignments = kmeans.fit_predict(gallery_features)
    index_table = {i: np.where(cluster_assignments == i)[0] for i in range(n_clusters)}
    return index_table, kmeans.cluster_centers_

def build_agglomerative_index(gallery_features, n_clusters):
    agg = AgglomerativeClustering(n_clusters=n_clusters, metric='euclidean', linkage='ward')
    cluster_assignments = agg.fit_predict(gallery_features)
    index_table = {i: np.where(cluster_assignments == i)[0] for i in range(n_clusters)}
    return index_table


# ------------------------------------------------------------
# 4. Evaluate Retrieval Performance
# ------------------------------------------------------------
def evaluate_hnsw_hit_rate(index, index_labels, query_vectors, query_labels, k_max=50):
    total_indexed = len(index_labels)
    total_queries = len(query_labels)
    distances, indices = index.search(query_vectors, k=k_max)
    predicted_labels = index_labels[indices]
    results = {'penetration_rates': [], 'hit_rates': []}

    for k in range(1, k_max + 1):
        hits = sum(query_labels[i] in predicted_labels[i, :k] for i in range(total_queries))
        hit_rate = hits / total_queries
        penetration = k / total_indexed
        results['hit_rates'].append(hit_rate)
        results['penetration_rates'].append(penetration)
    return results


def evaluate_cluster_index(probe_features, probe_labels, gallery_features, gallery_labels,
                           index_table, cluster_centers=None):
    n_clusters = len(index_table)
    total_gallery = len(gallery_features)
    results = {'penetration_rates': [], 'hit_rates': []}

    for n_search in range(1, n_clusters + 1):
        total_hits = 0
        total_candidates = 0
        for i, q in enumerate(probe_features):
            true_label = probe_labels[i]
            if cluster_centers is not None:
                dists = np.linalg.norm(cluster_centers - q, axis=1)
            else:
                dists = np.array([
                    np.linalg.norm(gallery_features[idxs] - q, axis=1).mean()
                    for idxs in index_table.values()
                ])
            nearest_clusters = np.argsort(dists)[:n_search]
            candidate_indices = np.concatenate([index_table[c] for c in nearest_clusters])
            total_candidates += len(candidate_indices)
            if len(candidate_indices) > 0:
                cand_labels = gallery_labels[candidate_indices]
                pred_label = cand_labels[np.argmin(np.linalg.norm(
                    gallery_features[candidate_indices] - q, axis=1))]
                if pred_label == true_label:
                    total_hits += 1
        hit_rate = total_hits / len(probe_features)
        penetration = (total_candidates / len(probe_features)) / total_gallery
        results['hit_rates'].append(hit_rate)
        results['penetration_rates'].append(penetration)
    return results


# ------------------------------------------------------------
# 5. Plot Comparison
# ------------------------------------------------------------
def plot_performance_curves(results_dict, title):
    fig = go.Figure()
    colors = {'HNSW': '#2ca02c', 'K-Means++': '#d62728', 'Agglomerative': '#1f77b4'}
    for method, results in results_dict.items():
        fig.add_trace(go.Scatter(
            x=results['penetration_rates'],
            y=results['hit_rates'],
            mode='lines+markers',
            name=method,
            line=dict(width=2, color=colors.get(method)),
            marker=dict(size=6)
        ))

    fig.update_layout(
        title=f"IITD_UPD Indexing Comparison — {title}",
        xaxis_title="Penetration Rate",
        yaxis_title="Hit Rate",
        xaxis=dict(tickvals=[i / 100 for i in range(0, 101, 1)],
                   ticktext=[f"{i/100:.2f}" for i in range(0, 101, 1)]),
        template="plotly_white",
        hovermode="x unified",
        legend=dict(x=0.02, y=0.98)
    )
    fig.show()


# ------------------------------------------------------------
# 6. Main Workflow
# ------------------------------------------------------------
def main():
    data_folder = "IITD_UPD"
    efSearch = 20
    k_max = 50
    N_CLUSTERS_KMEANS = 15
    N_CLUSTERS_AGG = 20

    index_x, index_y, query_x, query_y = load_iitd_upd_templates(data_folder)
    dim = index_x.shape[1]

    print(f"\nBuilding indexes and evaluating on dim={dim} feature vectors...")

    # --- HNSW ---
    hnsw_index = build_hnsw_index(index_x, dim, efSearch=efSearch)
    hnsw_results = evaluate_hnsw_hit_rate(hnsw_index, index_y, query_x, query_y, k_max=k_max)

    # --- KMeans ---
    kmeans_index, kmeans_centers = build_kmeans_index(index_x, n_clusters=N_CLUSTERS_KMEANS)
    kmeans_results = evaluate_cluster_index(query_x, query_y, index_x, index_y,
                                            kmeans_index, kmeans_centers)

    # --- Agglomerative ---
    agg_index = build_agglomerative_index(index_x, n_clusters=N_CLUSTERS_AGG)
    agg_results = evaluate_cluster_index(query_x, query_y, index_x, index_y, agg_index)

    # --- Plot ---
    plot_performance_curves({
        'HNSW': hnsw_results,
        'K-Means++': kmeans_results,
        'Agglomerative': agg_results
    }, "Hit Rate vs Penetration Rate")


if __name__ == "__main__":
    main()


✅ Loaded 1357 index and 883 query samples from 435 subjects.

Building indexes and evaluating on dim=1024 feature vectors...


In [17]:
import os
import numpy as np
import random
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")


# ---------------------------------------------------------
# Load IITD_UPD .npy templates (70:30 split)
# ---------------------------------------------------------
def load_iitd_upd_templates(root_folder, seed=42, split_ratio=0.7):
    random.seed(seed)
    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    subject_folders = sorted(
        [d for d in os.listdir(root_folder) if os.path.isdir(os.path.join(root_folder, d))]
    )

    print(f"\n📂 Found {len(subject_folders)} subject folders.")
    for subject in subject_folders:
        subject_path = os.path.join(root_folder, subject)
        files = sorted([f for f in os.listdir(subject_path) if f.endswith(".npy")])

        n = len(files)
        if n == 0:
            print(f"⚠️ Skipping {subject}, no .npy files.")
            continue

        random.shuffle(files)
        split_point = max(1, int(n * split_ratio))
        index_files = files[:split_point]
        query_files = files[split_point:] if split_point < n else []

        for f in index_files:
            vec = np.load(os.path.join(subject_path, f)).astype("float32").flatten()
            index_vectors.append(vec)
            index_labels.append(subject)

        for f in query_files:
            vec = np.load(os.path.join(subject_path, f)).astype("float32").flatten()
            query_vectors.append(vec)
            query_labels.append(subject)

    print(f"\n✅ Loaded {len(index_vectors)} index vectors and {len(query_vectors)} queries.")
    return (
        np.vstack(index_vectors),
        np.array(index_labels),
        np.vstack(query_vectors),
        np.array(query_labels),
    )


# ---------------------------------------------------------
# Build KMeans index
# ---------------------------------------------------------
def build_kmeans_index(gallery_features, n_clusters):
    print(f"\n⚙️ Building K-Means index with {n_clusters} clusters...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_assignments = kmeans.fit_predict(gallery_features)

    index_table = {i: [] for i in range(n_clusters)}
    for i, cid in enumerate(cluster_assignments):
        index_table[cid].append(i)

    print("✅ K-Means indexing complete.")
    return index_table, kmeans.cluster_centers_


# ---------------------------------------------------------
# Evaluate retrieval (Hit rate vs Penetration rate)
# ---------------------------------------------------------
def evaluate_retrieval(probe_features, probe_labels, gallery_features, gallery_labels,
                       index_table, cluster_centers):
    print("\n🔍 Evaluating retrieval across top-k clusters...")
    n_clusters = len(index_table)
    total_gallery = len(gallery_features)

    results = {"penetration_rates": [], "hit_rates": []}
    penetration_labels = []

    for n_search in range(1, n_clusters + 1):
        total_hits, total_searched = 0, 0

        for i, probe_vec in enumerate(probe_features):
            true_label = probe_labels[i]

            # Distance to each centroid
            dists = np.linalg.norm(cluster_centers - probe_vec, axis=1)
            nearest_clusters = np.argsort(dists)[:n_search]

            candidate_idxs = []
            for cid in nearest_clusters:
                candidate_idxs.extend(index_table[cid])

            total_searched += len(candidate_idxs)
            if not candidate_idxs:
                continue

            candidate_labels = gallery_labels[candidate_idxs]
            search_dists = np.linalg.norm(gallery_features[candidate_idxs] - probe_vec, axis=1)
            predicted_label = candidate_labels[np.argmin(search_dists)]

            if predicted_label == true_label:
                total_hits += 1

        hit_rate = total_hits / len(probe_features)
        penetration_rate = (total_searched / len(probe_features)) / total_gallery

        results["hit_rates"].append(hit_rate)
        results["penetration_rates"].append(penetration_rate)

        # Label only for 0.01, 0.02, 0.03 …
        if abs(round(penetration_rate * 100) % 1) < 1e-6 or round(penetration_rate * 100, 2) % 1 == 0:
            penetration_labels.append(round(penetration_rate, 2))
        else:
            penetration_labels.append(None)

        if n_search % 2 == 0 or n_search == n_clusters:
            print(f"Top-{n_search}: Hit={hit_rate:.3f}, Penetration={penetration_rate:.3f}")

    return results, penetration_labels


# ---------------------------------------------------------
# Plotting
# ---------------------------------------------------------
def plot_performance_curve(results, penetration_labels):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=results["penetration_rates"],
        y=results["hit_rates"],
        mode="lines+markers",
        name="K-Means",
        line=dict(width=2, color="#d62728"),
        marker=dict(size=6),
    ))

    fig.update_layout(
        title="K-Means Retrieval Performance (IITD_UPD)",
        xaxis_title="Penetration Rate",
        yaxis_title="Hit Rate",
        template="plotly_white",
        xaxis=dict(
            tickmode="array",
            tickvals=[x for x in results["penetration_rates"]
                      if round(x * 100, 2) in [i for i in range(1, 101, 1)]],
            tickformat=".0%",
        ),
        yaxis=dict(tickformat=".0%"),
        hovermode="x unified",
    )

    fig.show()


# ---------------------------------------------------------
# Main function
# ---------------------------------------------------------
def main():
    data_folder = "IITD_UPD"  # 🔹 Folder containing subject folders with .npy files
    n_clusters = 30            # Adjust this based on dataset size

    index_x, index_y, query_x, query_y = load_iitd_upd_templates(data_folder)
    print(f"Feature dimension = {index_x.shape[1]}")

    index_table, centers = build_kmeans_index(index_x, n_clusters)
    results, penetration_labels = evaluate_retrieval(query_x, query_y, index_x, index_y, index_table, centers)
    plot_performance_curve(results, penetration_labels)


# ---------------------------------------------------------
if __name__ == "__main__":
    main()



📂 Found 435 subject folders.

✅ Loaded 1357 index vectors and 883 queries.
Feature dimension = 1024

⚙️ Building K-Means index with 30 clusters...
✅ K-Means indexing complete.

🔍 Evaluating retrieval across top-k clusters...
Top-2: Hit=0.822, Penetration=0.076
Top-4: Hit=0.887, Penetration=0.153
Top-6: Hit=0.900, Penetration=0.228
Top-8: Hit=0.904, Penetration=0.302
Top-10: Hit=0.904, Penetration=0.373
Top-12: Hit=0.906, Penetration=0.444
Top-14: Hit=0.906, Penetration=0.515
Top-16: Hit=0.906, Penetration=0.582
Top-18: Hit=0.906, Penetration=0.649
Top-20: Hit=0.906, Penetration=0.714
Top-22: Hit=0.906, Penetration=0.776
Top-24: Hit=0.906, Penetration=0.836
Top-26: Hit=0.906, Penetration=0.891
Top-28: Hit=0.906, Penetration=0.943
Top-30: Hit=0.906, Penetration=1.000
